# Week 02 · Day 03 — AI Agents

**Topics:** ReAct loop from scratch · Tool registration · Memory strategies · Agent guards


In [ ]:
%pip install -q openai

In [ ]:
import os, json, re
from openai import OpenAI
from collections import deque

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 1 · Tool Registration

A tool registry maps tool names to (function, schema) pairs. The agent only knows about tools that are registered.

In [ ]:
from typing import Callable
from pydantic import BaseModel, Field

class ToolSpec(BaseModel):
    name: str
    description: str
    parameters: dict

TOOLS: dict[str, tuple[Callable, ToolSpec]] = {}

def register_tool(name: str, description: str, parameters: dict):
    def decorator(fn: Callable):
        TOOLS[name] = (fn, ToolSpec(name=name, description=description, parameters=parameters))
        return fn
    return decorator

# Register tools
@register_tool(
    name="search_web",
    description="Search the web for current information on a topic",
    parameters={"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
)
def search_web(query: str) -> str:
    # Stub — replace with real search API (Tavily, SerpAPI, etc.)
    return f"Search results for '{query}': [Result 1: Python was created in 1991. Result 2: Python 3.12 was released in 2023.]"

@register_tool(
    name="calculate",
    description="Evaluate a mathematical expression and return the result",
    parameters={"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]},
)
def calculate(expression: str) -> str:
    try:
        # Safe eval: only allow numbers and operators
        if re.search(r"[a-zA-Z_]", expression):
            return "ERROR: only numeric expressions allowed"
        return str(eval(expression))  # noqa: S307
    except Exception as e:
        return f"ERROR: {e}"

@register_tool(
    name="get_current_date",
    description="Get today's date",
    parameters={"type": "object", "properties": {}},
)
def get_current_date() -> str:
    from datetime import date
    return str(date.today())

print(f"Registered tools: {list(TOOLS.keys())}")

## 2 · ReAct Loop from Scratch

ReAct (Reason + Act) is the core agent loop:
1. **Thought**: the model reasons about what to do next
2. **Action**: the model calls a tool
3. **Observation**: the tool result is returned
4. Repeat until **Final Answer** is reached

In [ ]:
def build_tool_schemas() -> list[dict]:
    return [
        {"type": "function", "function": {
            "name": spec.name,
            "description": spec.description,
            "parameters": spec.parameters,
        }}
        for _, spec in TOOLS.values()
    ]

def run_agent(task: str, max_steps: int = 8, verbose: bool = True) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful agent. Use tools to answer questions. "
                "Think step by step. When you have the final answer, respond directly without calling tools."
            ),
        },
        {"role": "user", "content": task},
    ]

    tool_schemas = build_tool_schemas()

    for step in range(max_steps):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tool_schemas,
            temperature=0,
        )
        msg = response.choices[0].message

        # No tool calls — final answer
        if not msg.tool_calls:
            if verbose:
                print(f"[Step {step+1}] Final Answer")
            return msg.content

        messages.append(msg)

        for tc in msg.tool_calls:
            fn_name = tc.function.name
            args = json.loads(tc.function.arguments)

            if fn_name not in TOOLS:
                result = f"ERROR: Unknown tool '{fn_name}'"
            else:
                fn, _ = TOOLS[fn_name]
                try:
                    result = fn(**args)
                except Exception as e:
                    result = f"ERROR: {e}"

            if verbose:
                print(f"[Step {step+1}] {fn_name}({args}) → {str(result)[:80]}")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Agent hit max steps without finding an answer."

# Test the agent
print("=" * 60)
result = run_agent("What is 15% of 847 + the result of 23 * 17?")
print(f"\nAnswer: {result}")

In [ ]:
print("=" * 60)
result = run_agent("When was Python first released, and how many years ago was that from today?")
print(f"\nAnswer: {result}")

## 3 · Sliding Window Memory

Keep only the last N messages to prevent context overflow in long conversations.

In [ ]:
class SlidingWindowAgent:
    def __init__(self, window_size: int = 10, max_steps: int = 8):
        self.window_size = window_size
        self.max_steps = max_steps
        self.history = deque(maxlen=window_size)
        self.system_prompt = (
            "You are a helpful assistant. Use tools when needed. "
            "When you have the final answer, respond without tool calls."
        )

    def _get_messages(self, user_input: str) -> list:
        return [
            {"role": "system", "content": self.system_prompt},
            *list(self.history),
            {"role": "user", "content": user_input},
        ]

    def chat(self, user_input: str) -> str:
        self.history.append({"role": "user", "content": user_input})
        messages = self._get_messages(user_input)[:-1]  # exclude duplicate user msg

        for _ in range(self.max_steps):
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                tools=build_tool_schemas(),
                temperature=0.3,
            )
            msg = response.choices[0].message

            if not msg.tool_calls:
                self.history.append({"role": "assistant", "content": msg.content})
                return msg.content

            messages.append(msg)
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                fn, _ = TOOLS.get(tc.function.name, (lambda **kw: "Unknown tool", None))
                result = fn(**args) if tc.function.name in TOOLS else "Unknown tool"
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})

        return "Could not complete the task."

agent = SlidingWindowAgent(window_size=10)

turns = [
    "Hi, my name is Alice.",
    "What is 2 to the power of 10?",
    "Can you remind me what my name is?",
]

for turn in turns:
    response = agent.chat(turn)
    print(f"Human: {turn}")
    print(f"Agent: {response}\n")

## 4 · Agent Guards (Preventing Runaway Loops)

In [ ]:
import time
from collections import Counter

def guarded_agent(task: str, max_steps: int = 10, timeout_s: float = 30.0) -> str:
    messages = [
        {"role": "system", "content": "Use tools to answer. Stop when you have the final answer."},
        {"role": "user", "content": task},
    ]
    tool_call_counts = Counter()
    start = time.time()

    for step in range(max_steps):
        # Timeout guard
        if time.time() - start > timeout_s:
            return f"TIMEOUT after {timeout_s}s"

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=build_tool_schemas(),
            temperature=0,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content

        messages.append(msg)

        for tc in msg.tool_calls:
            tool_call_counts[tc.function.name] += 1

            # Repetition guard — same tool called more than 3 times
            if tool_call_counts[tc.function.name] > 3:
                return f"LOOP DETECTED: '{tc.function.name}' called {tool_call_counts[tc.function.name]} times"

            args = json.loads(tc.function.arguments)
            fn, _ = TOOLS.get(tc.function.name, (None, None))
            result = fn(**args) if fn else "ERROR: Unknown tool"
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})

    return f"MAX STEPS ({max_steps}) reached"

# Normal task
print(guarded_agent("What is today's date?"))

## Summary

- **ReAct**: Thought → Action → Observation loop. Model reasons, you execute.
- **Tool registry**: name → (function, schema). Only registered tools are visible to the agent.
- **Sliding window**: `deque(maxlen=N)` keeps the last N messages without manual truncation.
- **Guards**: max_steps + timeout + repetition counter prevent runaway loops.
- Always validate tool arguments before executing; return errors as strings (never raise).

**Next:** [LangGraph](week02-day03-langgraph.ipynb)